# BitTokens: An Interactive Introduction

This notebook introduces the core contribution of **BitTokens** in three steps:

- Traditional tokenizers break numbers into many pieces.
- BitTokens keeps numbers as a dedicated single token pathway.
- The model uses an efficient bit-level encoding for training and inference.

## What you will see

1. Side-by-side tokenization for single-digit, subword, and bittokens.
2. How `BitTokenEmbedding` encodes numbers and combines embeddings.
3. How the linear numeric head predicts bits, computes training loss, and decodes numbers.

In [1]:
import re
from dataclasses import dataclass
from pathlib import Path

import torch
from IPython.display import HTML, display
from transformers import PreTrainedTokenizerFast

from utils.util_funcs import NUMBER_PARSE_REGEX

# Repository root (this notebook is expected to live at repo root)
REPO_ROOT = Path('.').resolve()

In [2]:

NUMERIC_SPAN_REGEX = re.compile(NUMBER_PARSE_REGEX)

TOKENIZER_DIRS = {
    'sd_gpt2 (single-digit baseline)': REPO_ROOT / 'tokenizers/num_text/sd_gpt2',
    'td_gpt2 (subword baseline)': REPO_ROOT / 'tokenizers/num_text/td_gpt2',
    'bittoken_gpt2 (BitTokens)': REPO_ROOT / 'tokenizers/num_text/bittoken_gpt2',
}


def load_tokenizer(tokenizer_dir: Path) -> PreTrainedTokenizerFast:
    tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_dir)
    tokenizer.padding_side = 'left'
    return tokenizer


def load_all_tokenizers() -> dict[str, PreTrainedTokenizerFast]:
    return {name: load_tokenizer(path) for name, path in TOKENIZER_DIRS.items()}


def html_escape(s: str) -> str:
    return (
        s.replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
        .replace('"', '&quot;')
    )


def looks_numeric_piece(token: str) -> bool:
    # Heuristic for traditional tokenizers where numbers are split across pieces.
    # Dot is intentionally excluded here and handled with neighbor context.
    return bool(re.search(r'[-+]?\d|e\+|e\-|E\+|E\-', token))


def is_numeric_dot(tokens: list[str], idx: int) -> bool:
    if tokens[idx] != '.':
        return False
    prev_has_digit = idx > 0 and bool(re.search(r'\d', tokens[idx - 1]))
    next_has_digit = idx < len(tokens) - 1 and bool(re.search(r'\d', tokens[idx + 1]))
    return prev_has_digit and next_has_digit


tokenizers = load_all_tokenizers()
print('Loaded tokenizers:', ', '.join(tokenizers.keys()))

Loaded tokenizers: sd_gpt2 (single-digit baseline), td_gpt2 (subword baseline), bittoken_gpt2 (BitTokens)


## 1) Tokenization comparison: SD vs TD vs BitTokens

Traditional tokenizers often split long numbers into many pieces. The BitToken tokenizer uses dedicated number tokens so numeric text can be represented more compactly.

This section tokenizes the same text with three tokenizers and compares:

- token pieces
- token IDs
- numeric-focused tokens (highlighted)
- total token counts

Related code:

- [`analyze_token_counts.py`](analyze_token_counts.py)
- [`utils/util_funcs.py`](utils/util_funcs.py)
- [`dataloader/datasets/pretokenized_number_dataset.py`](dataloader/datasets/pretokenized_number_dataset.py)

In [3]:
sample_text = (
    'Sensor A reported 1234567890.123456, while sensor B dropped to -0.0000003141592653. '
    'A third run produced 602214076000000000000000.0 and 9.109383701531 in the same line.'
)
raw_numbers = [m.group(0).strip() for m in NUMERIC_SPAN_REGEX.finditer(sample_text)]

@dataclass
class TokenizationView:
    name: str
    ids: list[int]
    tokens: list[str]
    numeric_mask: list[bool]


def get_num_token_ids_for_bittoken(tokenizer: PreTrainedTokenizerFast) -> list[int]:
    num_ids: list[int] = []

    # Newer tokenizers may store a dedicated numeric token id.
    num_token_id = tokenizer.init_kwargs.get('num_token_id')
    if isinstance(num_token_id, int):
        num_ids.append(num_token_id)

    # Older setups may store one or more numeric token strings.
    num_tokens = tokenizer.init_kwargs.get('num_token', [])
    if isinstance(num_tokens, str):
        num_tokens = [num_tokens]
    num_ids.extend(tokenizer.convert_tokens_to_ids(t) for t in num_tokens)

    # Fallback for display notebooks: explicit known placeholder.
    placeholder_id = tokenizer.convert_tokens_to_ids('<|num|>')
    if placeholder_id >= 0:
        num_ids.append(placeholder_id)

    return sorted(set(num_ids))


def replace_numbers_with_num_token(text: str, num_token: str = '<|num|>') -> str:
    # Reuse repo number-parse schema (NUMBER_PARSE_REGEX) for replacement.
    return re.sub(NUMERIC_SPAN_REGEX, num_token, text)


def tokenize_for_view(name: str, tokenizer: PreTrainedTokenizerFast, text: str) -> TokenizationView:
    text_for_tokenization = text

    if 'bittoken' in name:
        # Match training-time behavior conceptually: replace full numeric spans by num token.
        text_for_tokenization = replace_numbers_with_num_token(text)

    encoded = tokenizer(text_for_tokenization, padding='do_not_pad', return_tensors=None)
    ids = encoded['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(ids)

    if 'bittoken' in name:
        num_ids = set(get_num_token_ids_for_bittoken(tokenizer))
        numeric_mask = [token_id in num_ids for token_id in ids]
    else:
        numeric_mask = [
            looks_numeric_piece(tok) or is_numeric_dot(tokens, i)
            for i, tok in enumerate(tokens)
        ]

    return TokenizationView(name=name, ids=ids, tokens=tokens, numeric_mask=numeric_mask)


def highlight_token(tok: str, is_numeric: bool) -> str:
    style = (
        'background:#ffe082; color:#000; padding:2px 4px; margin:1px; border-radius:4px; display:inline-block;'
        if is_numeric else
        'background:#eceff1; color:#000; padding:2px 4px; margin:1px; border-radius:4px; display:inline-block;'
    )
    return f"<span style='{style}'>{html_escape(tok)}</span>"


def render_token_panel(view: TokenizationView) -> str:
    token_html = ''.join(highlight_token(t, m) for t, m in zip(view.tokens, view.numeric_mask))
    total = len(view.ids)
    numeric_count = int(sum(view.numeric_mask))
    return (
        f"<h4>{html_escape(view.name)}</h4>"
        f"<div style='font-size:13px; line-height:1.8;'>{token_html}</div>"
        f"<p><b>Total tokens:</b> {total} &nbsp; | &nbsp; <b>Numeric-focused tokens:</b> {numeric_count}</p>"
    )


views = [tokenize_for_view(name, tok, sample_text) for name, tok in tokenizers.items()]
panels = ''.join(
    f"<div style='flex:1; min-width:280px; border:1px solid #ddd; padding:10px; border-radius:8px;'>{render_token_panel(v)}</div>"
    for v in views
)

display(HTML(f"<div style='display:flex; gap:12px; flex-wrap:wrap;'>{panels}</div>"))

### Why this matters

- Fewer tokens for numeric-heavy text means more effective context length.
- Numeric values can be handled by a dedicated pathway instead of only subword pieces.
- This separation helps the model treat numeric magnitude and language context differently.

## 2) How BitToken embedding works

Relevant implementation paths:

- `StemHeadModel.compute_input_embeddings`: [`networks/stem_head_model.py`](networks/stem_head_model.py)
- `BitTokenEmbedding.forward` and `combine_embeds`: [`networks/number_embedding_modules/bittoken_embedding.py`](networks/number_embedding_modules/bittoken_embedding.py)

At a high level:

1. Detect number-token positions (`number_mask`).
2. Encode only those numeric values into bit vectors.
3. Combine token embeddings and numeric encodings at masked positions.

The embedding path reinterprets `float64` storage as `int64` to access the IEEE-754 bit pattern directly, then extracts bits with vectorized tensor ops.

In [4]:
float64_bit_shifts: torch.LongTensor = torch.arange(64-1, -1, -1, dtype=torch.int64)

def float64_tensor_to_binary_tensor(tensor_in: torch.DoubleTensor) -> torch.LongTensor:
    """
    Converts a float64 PyTorch tensor to its IEEE 754 binary representation,
    returning the result as a new PyTorch integer tensor of bits.

    Args:
        tensor_in (torch.DoubleTensor): A PyTorch tensor with dtype torch.float64.

    Returns:
        base_extension (torch.LongTensor): A PyTorch tensor of dtype torch.int64 with shape (*tensor_in.shape, 64)
    """
    int_representation = tensor_in.view(torch.int64).unsqueeze(-1)
    bits = (int_representation >> float64_bit_shifts) & 1
    return bits

nums = torch.tensor([float(n) for n in raw_numbers], dtype=torch.float64)
bits = float64_tensor_to_binary_tensor(nums)

print('Numbers extracted from sample_text:', [float(n) for n in raw_numbers])
print('Input shape:', tuple(nums.shape))
print('Bit encoding shape (64-bit):', tuple(bits.shape))

# Optional reciprocal encoding (used optionally in BitTokenEmbedding)
reciprocal_bits = float64_tensor_to_binary_tensor(nums.reciprocal())
full_encoding = torch.cat([bits, reciprocal_bits], dim=-1)
print('With reciprocal concat shape (128-bit):', tuple(full_encoding.shape))

Numbers extracted from sample_text: [1234567890.123456, -3.141592653e-07, 6.02214076e+23, 9.109383701531]
Input shape: (4,)
Bit encoding shape (64-bit): (4, 64)
With reciprocal concat shape (128-bit): (4, 128)


In [5]:
# Visualize each 64-bit stream as colored chips (sign / exponent / mantissa)
def render_ieee754_bits_row(value: float, bit_row: torch.Tensor) -> str:
    bit_chars = [str(int(b)) for b in bit_row.tolist()]

    def chip(bit: str, bg: str) -> str:
        return (
            f"<span style='background:{bg}; color:#000; padding:2px 4px; margin:1px; "
            "border-radius:4px; display:inline-block; font-family:monospace;'>"
            f"{bit}</span>"
        )

    sign_bits = ''.join(chip(b, '#ffcc80') for b in bit_chars[:1])
    exponent_bits = ''.join(chip(b, '#90caf9') for b in bit_chars[1:12])
    mantissa_bits = ''.join(chip(b, '#c5e1a5') for b in bit_chars[12:])

    return (
        "<div style='margin-bottom:10px;'>"
        f"<div><b>value:</b> {value}</div>"
        "<div style='line-height:1.8;'>"
        f"{sign_bits} {exponent_bits} {mantissa_bits}"
        "</div>"
        "</div>"
    )


def render_ieee754_bits_table(values: torch.Tensor, bits_tensor: torch.Tensor) -> None:
    legend = (
        "<div style='margin-bottom:10px;'>"
        "<span style='background:#ffcc80; padding:2px 6px; margin-right:8px; color:#000; border-radius:4px;'>sign (1b)</span>"
        "<span style='background:#90caf9; padding:2px 6px; margin-right:8px; color:#000; border-radius:4px;'>exponent (11b)</span>"
        "<span style='background:#c5e1a5; padding:2px 6px; color:#000; border-radius:4px;'>mantissa (52b)</span>"
        "</div>"
    )

    rows = ''.join(
        render_ieee754_bits_row(float(values[i].item()), bits_tensor[i])
        for i in range(values.shape[0])
    )

    display(HTML(legend + rows))


render_ieee754_bits_table(nums, bits)

`StemHeadModel.compute_input_embeddings` follows this flow:

- build normal token embeddings
- compute `number_mask` from number token IDs
- encode numeric values only at masked positions
- combine token + numeric vectors into `combined_embeds`

That keeps non-numeric token processing unchanged, while enriching numeric positions.

In [ ]:
# Combine 128-bit number encoding with 384-d text embedding from sample_text
def combine_embeddings(
    inputs_embeds: torch.Tensor,
    num_encoding: torch.Tensor,
    number_mask: torch.Tensor,
) -> torch.Tensor:
    combined = inputs_embeds.clone()
    n_embed = inputs_embeds.shape[-1]
    emb_size = num_encoding.shape[-1]
    pad_size = n_embed - emb_size

    num_encoding = num_encoding.to(inputs_embeds.dtype)
    num_encoding = num_encoding.clone()
    num_encoding[number_mask] = num_encoding[number_mask] * 2 - 1  # {0,1} -> {-1,1}

    padded = torch.nn.functional.pad(num_encoding[number_mask], (0, pad_size), value=0.0)
    combined[number_mask] = inputs_embeds[number_mask] + padded
    return combined

# Build token embeddings from the same sample_text pipeline.
bittoken_tokenizer = tokenizers['bittoken_gpt2 (BitTokens)']
text_for_bittoken = replace_numbers_with_num_token(sample_text)
encoded = bittoken_tokenizer(text_for_bittoken, padding='do_not_pad', return_tensors=None)
input_ids = torch.tensor(encoded['input_ids'], dtype=torch.long).unsqueeze(0)  # (1, S)

# 384-dimensional text embedding
# Use len(tokenizer) instead of vocab_size for robust embedding table size.
vocab_size = int(len(bittoken_tokenizer))
assert vocab_size > 0, f'Invalid tokenizer size: {vocab_size}'
assert int(input_ids.min()) >= 0, f'Negative token id found: {int(input_ids.min())}'
assert int(input_ids.max()) < vocab_size, (
    f'Token id {int(input_ids.max())} is out of range for vocab_size={vocab_size}'
)

embedding_layer = torch.nn.Embedding(vocab_size, 384)
inputs_embeds = embedding_layer(input_ids)  # (1, S, 384)

# Numeric token mask in sequence
num_token_ids = torch.tensor(get_num_token_ids_for_bittoken(bittoken_tokenizer), dtype=torch.long)
number_mask = torch.isin(input_ids, num_token_ids)
num_positions = int(number_mask.sum().item())

num_encoding = torch.zeros((1, input_ids.shape[1], 128), dtype=torch.float32)
num_encoding[number_mask] = full_encoding[:num_positions].float()

combined = combine_embeddings(inputs_embeds, num_encoding, number_mask)

print('text_for_bittoken:', text_for_bittoken)
print('vocab_size:', vocab_size)
print('input_ids shape:', tuple(input_ids.shape))
print('inputs_embeds shape:', tuple(inputs_embeds.shape))
print('num_encoding shape:', tuple(num_encoding.shape))
print('number_mask shape:', tuple(number_mask.shape), '| numeric positions:', num_positions)
print('combined_embeds shape:', tuple(combined.shape))

text_for_bittoken: Sensor A reported <|num|>, while sensor B dropped to <|num|>. A third run produced <|num|> and <|num|> in the same line.
vocab_size: 48577
input_ids shape: (1, 28)
inputs_embeds shape: (1, 28, 384)
num_encoding shape: (1, 28, 128)
number_mask shape: (1, 28) | numeric positions: 4
combined_embeds shape: (1, 28, 384)


## 3) Numeric head, training loss, and inference decode

The numeric branch is configured with Linear layer number head.

Conceptual path:

- hidden state at numeric token positions -> linear head -> predicted bits
- training: compare predictions to target bit encodings (`compute_num_loss`)
- inference: threshold predicted bits, then convert bits back to float (`decode`)

In [7]:
# Linear numeric head demo on the same sample_text pipeline
# Reuse `combined` and `number_mask` from the previous section.
hidden_states = combined  # (1, S, 384)
output_size = bits.shape[-1]  # 128 bits (64 + reciprocal 64)

num_head_linear = torch.nn.Linear(hidden_states.shape[-1], output_size)
pred_bits_logits = num_head_linear(hidden_states[number_mask])

print('Hidden states shape:', tuple(hidden_states.shape))
print('Masked numeric positions:', int(number_mask.sum()))
print('Predicted bit-logits shape:', tuple(pred_bits_logits.shape))

Hidden states shape: (1, 28, 384)
Masked numeric positions: 4
Predicted bit-logits shape: (4, 64)


In [8]:
# Training-loss demo inspired by compute_num_loss (same sample_text numbers)
# Targets come from the real 128-bit encoding of extracted sample_text numbers.
target_bits = bits.to(torch.float32)
assert target_bits.shape == pred_bits_logits.shape, (
    f'Target/pred shape mismatch: {target_bits.shape} vs {pred_bits_logits.shape}'
)

freq_weights = torch.ones(output_size).unsqueeze(0) # equal bit weighting

bce_per_bit = torch.nn.functional.binary_cross_entropy_with_logits(
    pred_bits_logits,
    target_bits,
    reduction='none',
)
weighted_bce = bce_per_bit * freq_weights
loss = weighted_bce.mean(dim=-1)

print('Target bits shape:', tuple(target_bits.shape))
print('Mean weighted BCE loss:', loss.mean().item())

Target bits shape: (4, 64)
Mean weighted BCE loss: 0.791466236114502


In [9]:
def binary_tensor_to_float64_tensor(bits_int64: torch.Tensor) -> torch.Tensor:
    """
    Reconstructs a float64 tensor from its IEEE 754 binary representation.
    This function is the inverse of `float64_tensor_to_binary_tensor`.
    Args:
        bits_int64: A PyTorch tensor with an integer dtype and shape (N, 64),
                    where the last dimension holds the 64 bits (0s or 1s)
                    of each float number, from most-significant to least-significant.
    Returns:
        num_tensor (torch.DoubleTensor): A tensor of reconstructed float64 values with shape (N).
    """
    exponents = float64_bit_shifts
    weights = torch.tensor(1, dtype=torch.int64) << exponents
    reconstructed_int = torch.sum(bits_int64 * weights, dim=-1)
    return reconstructed_int.to(dtype=torch.int64).view(torch.float64).to(torch.float64)

# Simulate a noisy prediction
pred_bits_logits = (bits*2-1) + torch.rand_like(bits.float()).sub_(0.5)

x_base_digits_pred: torch.LongTensor = (pred_bits_logits > 0).to(torch.int64)
num_preds = binary_tensor_to_float64_tensor(x_base_digits_pred)

# Compare to the original sample number that produced this numeric position.
target_value = nums[0]

print('Pred bits shape (first 64):', tuple(x_base_digits_pred.shape))
print('Decoded float value:', float(num_preds[0]))
print('Target sample_text value:', float(target_value))

Pred bits shape (first 64): (4, 64)
Decoded float value: 1234567890.123456
Target sample_text value: 1234567890.123456


## Benefits recap

- **D1 - Token efficiency:** Every number is represented by a single token. Fewer tokens for numeric-heavy text can preserve context for surrounding language.
- **D2 - Uniqueness:** Each value has exactly one valid encoding, with a unique inverse mapping.
- **D3 - Structured:** The encoding geometry reflects numeric order and distance, facilitating generalizable algorithms.
- **D4 - Scale invariance:** A large range of input magnitudes and precisions can be represented.
- **D5 - Normalization:** Encodings are bounded and information preserving under standard normalization functions used in language models (e.g., LayerNorm, RMSNorm)
- **D6 - Numerical stability:** Representations remain accurate when using low-precision activations (e.g., FP8)
- **D7 - Continuity:** Encodings vary relatively smoothly with the underlying value, making them compatible with gradient-based optimization
- **D8 - Robustness:** Values can be decoded reliably under stochastic noise, allowing for stochastic training.
- **D9 - Arithmetic:** Encodings admit learnable algorithms for core mathematical operations.

## Where to read next in the codebase

1. [`networks/number_embedding_modules/bittoken_embedding.py`](networks/number_embedding_modules/bittoken_embedding.py)
2. [`networks/stem_head_model.py`](networks/stem_head_model.py)
3. [`tokenizers/create_tokenizers.py`](tokenizers/create_tokenizers.py)